# Model Tambahan: Menebak Kategori dan Rating dari Teks Ulasan

Notebook ini mencoba dua tugas lain yang disarankan untuk dataset ulasan:

- Bagian A, klasifikasi kategori: menebak kategori produk (elektronik, fashion, olahraga, handphone, pertukangan) hanya dari teks ulasan.
- Bagian B, prediksi rating: menebak bintang 1 sampai 5 dari teks ulasan.

Pola kerjanya sama dengan `06_model_sentimen.ipynb`: data dibagi dulu, pengaturan model dipilih dengan cross-validation di data latih, lalu model dinilai sekali di data uji.

Kedua model ini tidak dipakai di dashboard. Tujuannya menguji seberapa jauh teks ulasan bisa dipakai untuk tugas lain, dan membandingkan hasilnya dengan model sentimen dua kelas.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupShuffleSplit,
    StratifiedGroupKFold,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline

# Folder utama proyek dimasukkan ke daftar pencarian modul, supaya folder src bisa di-import
sys.path.append(str(Path("..").resolve()))
from src.teks import bersihkan_teks

In [ ]:
# Gaya grafik, sama dengan notebook sebelumnya
BIRU = "#2a78d6"
ORANYE = "#eb6834"
ABU = "#c8c6bf"
TEKS = "#52514e"

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#8a8983",
    "axes.labelcolor": TEKS,
    "xtick.color": TEKS,
    "ytick.color": TEKS,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
})

PETA_BIRU = LinearSegmentedColormap.from_list("biru", ["#f4f3ef", BIRU])


def gambar_confusion(matriks_persen, label, judul):
    fig, ax = plt.subplots(figsize=(5.8, 4.6))
    ax.imshow(matriks_persen, cmap=PETA_BIRU, vmin=0, vmax=100)
    ax.set_xticks(range(len(label)), labels=label, rotation=30, ha="right")
    ax.set_yticks(range(len(label)), labels=label)
    for i in range(len(label)):
        for j in range(len(label)):
            ax.text(j, i, f"{matriks_persen[i, j]:.0f}%", ha="center", va="center", fontsize=9, color="#0b0b0b")
    ax.set_xlabel("Tebakan model")
    ax.set_ylabel("Label asli")
    ax.grid(False)
    ax.set_title(judul)
    plt.tight_layout()
    plt.show()

## Menyiapkan data

Prediksi rating butuh ulasan bintang 3, padahal bintang 3 sudah dibuang dari `ulasan_bersih.csv`. Karena itu data diambil lagi dari file mentah dan dibersihkan ulang.

Fungsi `bersihkan_teks` kini disimpan di file `src/teks.py`, bukan disalin ulang ke notebook ini. Dengan begitu notebook ini, notebook-notebook lain, dan dashboard nanti memakai satu fungsi yang sama. Sel di bawah memastikan hasilnya sama persis dengan hasil notebook 05.

In [ ]:
df = pd.read_csv("../Dataset/raw/tokopedia-product-reviews-2019.csv")
df = df.drop_duplicates(subset=["text", "product_id"])
df["teks_bersih"] = df["text"].map(bersihkan_teks)
df = df[df["teks_bersih"].str.strip() != ""].reset_index(drop=True)
print(df.shape)

In [ ]:
# Hasil src/teks.py harus sama dengan hasil notebook 05
ulasan_bersih = pd.read_csv("../Dataset/processed/ulasan_bersih.csv")
cocok = df.merge(ulasan_bersih, left_on=["text", "product_id"], right_on=["teks", "id_produk"],
                 suffixes=("", "_nb05"))
assert (cocok["teks_bersih"] == cocok["teks_bersih_nb05"].fillna("")).all()
print(f"{len(cocok):,} ulasan diperiksa, semuanya sama.")

In [ ]:
def buat_pipeline(**pengaturan_model):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("model", LogisticRegression(max_iter=3000, **pengaturan_model)),
    ])


GRID = {
    "model__C": [1, 10],
    "model__class_weight": [None, "balanced"],
}

## Bagian A: Klasifikasi kategori

### A1. Membagi data per produk

Di notebook 06, data dibagi secara acak. Untuk tugas ini pembagian acak berisiko. Ulasan untuk produk yang sama sering menyebut nama atau ciri produk itu, misalnya "headsetnya" atau "sepatunya". Kalau ulasan dari produk yang sama tersebar di data latih dan data uji, model bisa sekadar menghafal produk, bukan belajar mengenali kategori.

Karena itu data dibagi per produk dengan `GroupShuffleSplit`: semua ulasan dari satu produk masuk ke data latih saja atau data uji saja. Cross-validation juga dibagi per produk dengan `StratifiedGroupKFold`.

In [ ]:
X = df["teks_bersih"]
y_kategori = df["category"]
produk = df["product_id"]

pembagi = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_latih, idx_uji = next(pembagi.split(X, y_kategori, groups=produk))

X_latih, X_uji = X.iloc[idx_latih], X.iloc[idx_uji]
y_latih, y_uji = y_kategori.iloc[idx_latih], y_kategori.iloc[idx_uji]
produk_latih = produk.iloc[idx_latih]

print("Data latih:", len(X_latih), "| Data uji:", len(X_uji))
print("Produk yang ada di keduanya:", len(set(produk_latih) & set(produk.iloc[idx_uji])))
pd.DataFrame({"latih": y_latih.value_counts(normalize=True), "uji": y_uji.value_counts(normalize=True)}).round(3)

### A2. Model patokan

Model patokan selalu menebak kategori terbanyak, yaitu elektronik.

Ukuran utama yang dipakai adalah macro F1, yaitu rata-rata F1 dari kelima kategori dengan bobot yang sama. Akurasi saja kurang adil, karena elektronik jauh lebih banyak dari kategori lain. Model yang pandai menebak elektronik tetapi gagal di pertukangan tetap bisa mendapat akurasi tinggi, sedangkan macro F1-nya akan rendah.

In [ ]:
patokan = DummyClassifier(strategy="most_frequent").fit(X_latih, y_latih)
tebakan = patokan.predict(X_uji)
print(f"Akurasi: {accuracy_score(y_uji, tebakan):.3f} | macro F1: {f1_score(y_uji, tebakan, average='macro'):.3f}")

### A3. Memilih pengaturan dengan GridSearchCV

In [ ]:
grid_kategori = GridSearchCV(
    buat_pipeline(),
    GRID,
    scoring="f1_macro",
    cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=2,
)
grid_kategori.fit(X_latih, y_latih, groups=produk_latih)

print("Pengaturan terbaik:", grid_kategori.best_params_)
print(f"Macro F1 cross-validation: {grid_kategori.best_score_:.3f}")

### A4. Evaluasi di data uji

In [ ]:
model_kategori = grid_kategori.best_estimator_
tebakan_kategori = model_kategori.predict(X_uji)
print(classification_report(y_uji, tebakan_kategori, digits=3))

In [ ]:
label_kategori = sorted(y_kategori.unique())
matriks = confusion_matrix(y_uji, tebakan_kategori, labels=label_kategori)
matriks_persen = matriks / matriks.sum(axis=1, keepdims=True) * 100
gambar_confusion(matriks_persen, label_kategori, "Kategori: persen per baris")

Setiap baris confusion matrix di atas dijumlahkan menjadi 100%. Baris "fashion", misalnya, menunjukkan ke mana saja ulasan fashion ditebak oleh model. Angka di diagonal adalah persentase yang ditebak benar.

### A5. Kata yang paling khas untuk setiap kategori

In [ ]:
kosakata = model_kategori.named_steps["tfidf"].get_feature_names_out()
bobot = model_kategori.named_steps["model"].coef_

kata_khas = pd.DataFrame({
    kategori: kosakata[np.argsort(bobot[i])[::-1][:10]]
    for i, kategori in enumerate(model_kategori.classes_)
})
kata_khas

### Kesimpulan bagian A

Klasifikasi kategori dari teks ulasan hanya berhasil sebagian:

- Akurasinya sekitar 41%, hampir sama dengan model patokan (39%). Yang jauh lebih baik adalah macro F1 (sekitar 0,35 dibanding 0,11), karena model mau menebak kategori selain elektronik. Namun precision untuk pertukangan hanya sekitar 6%, artinya hampir semua tebakan "pertukangan" salah.
- Penyebab utamanya terlihat dari data: sebagian besar ulasan membahas hal yang sama di semua kategori, seperti "barang sesuai pesanan" dan "pengiriman cepat", tanpa menyebut produknya. Model hanya bisa menebak dengan yakin kalau ulasan menyebut ciri produk, misalnya "sepatu" dan "ukuran" untuk fashion, atau "suara" dan "hp" untuk handphone.
- Kata khas yang dipelajari model juga berisi merek dan nama tempat, seperti "bata", "nokia", "krisbow", dan "surabaya". Data sudah dibagi per produk, tetapi toko dan merek yang sama tetap bisa muncul di data latih dan data uji. Jadi sebagian kemampuan model berasal dari menghafal merek atau toko, bukan memahami kategori.

Untuk menebak kategori, nama produk jauh lebih informatif daripada teks ulasan. Teks ulasan lebih cocok dipakai untuk menilai pengalaman pembeli, seperti di model sentimen.

## Bagian B: Prediksi rating

### B1. Membagi data

Pembagiannya sama dengan notebook 06, yaitu acak dengan `stratify` supaya porsi setiap bintang sama di data latih dan data uji. Tugas ini memakai semua bintang, termasuk bintang 3.

Selain akurasi dan macro F1, dipakai juga MAE (mean absolute error): rata-rata selisih bintang antara tebakan dan label asli. MAE 0,3 berarti rata-rata tebakan meleset 0,3 bintang. Ukuran ini berguna karena salah menebak bintang 4 sebagai 5 jauh lebih ringan daripada salah menebak bintang 1 sebagai 5.

In [ ]:
y_rating = df["rating"]
X_latih_r, X_uji_r, y_latih_r, y_uji_r = train_test_split(
    X, y_rating, test_size=0.2, stratify=y_rating, random_state=42
)
y_latih_r.value_counts(normalize=True).sort_index().round(3)

### B2. Model patokan

In [ ]:
def nilai_rating(asli, tebakan):
    return {
        "akurasi": accuracy_score(asli, tebakan),
        "macro_f1": f1_score(asli, tebakan, average="macro"),
        "mae": mean_absolute_error(asli, tebakan),
    }


patokan_r = DummyClassifier(strategy="most_frequent").fit(X_latih_r, y_latih_r)
hasil_rating = {"Patokan (selalu bintang 5)": nilai_rating(y_uji_r, patokan_r.predict(X_uji_r))}
pd.DataFrame(hasil_rating).T.round(3)

### B3. Memilih pengaturan dengan GridSearchCV

In [ ]:
grid_rating = GridSearchCV(
    buat_pipeline(),
    GRID,
    scoring="f1_macro",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=2,
)
grid_rating.fit(X_latih_r, y_latih_r)

print("Pengaturan terbaik:", grid_rating.best_params_)
print(f"Macro F1 cross-validation: {grid_rating.best_score_:.3f}")

### B4. Evaluasi di data uji

In [ ]:
model_rating = grid_rating.best_estimator_
tebakan_rating = model_rating.predict(X_uji_r)

hasil_rating["Logistic Regression (terpilih, class_weight balanced)"] = nilai_rating(y_uji_r, tebakan_rating)
print(classification_report(y_uji_r, tebakan_rating, digits=3))

Sebagai pembanding, model yang sama juga dilatih tanpa `class_weight`. Model ini tidak dipilih oleh GridSearchCV karena macro F1-nya sedikit lebih rendah, tetapi perbandingannya menunjukkan hal penting tentang pilihan ukuran.

In [ ]:
model_tanpa_bobot = buat_pipeline(C=grid_rating.best_params_["model__C"]).fit(X_latih_r, y_latih_r)
hasil_rating["Logistic Regression (tanpa class_weight)"] = nilai_rating(y_uji_r, model_tanpa_bobot.predict(X_uji_r))
pd.DataFrame(hasil_rating).T.round(3)

Hasilnya menunjukkan bahwa ukuran yang dipilih menentukan model mana yang terlihat "terbaik":

- Model terpilih (dengan `class_weight`) punya macro F1 tertinggi, tetapi akurasi dan MAE-nya justru lebih buruk daripada model patokan yang selalu menebak bintang 5. Model ini lebih berani menebak bintang rendah, sehingga lebih sering menemukan bintang 1, tetapi juga sering salah menurunkan ulasan bintang 5 menjadi 4 atau 3.
- Model tanpa `class_weight` punya akurasi dan MAE yang sedikit lebih baik dari patokan, tetapi menebak bintang 5 untuk sekitar 93% ulasan. Perilakunya hampir sama dengan model patokan.

Tidak ada satu pun model yang unggul di semua ukuran. Ini tanda bahwa tugasnya sendiri memang sulit dijawab dari teks ulasan.

In [ ]:
label_rating = [1, 2, 3, 4, 5]
matriks_r = confusion_matrix(y_uji_r, tebakan_rating, labels=label_rating)
matriks_r_persen = matriks_r / matriks_r.sum(axis=1, keepdims=True) * 100
gambar_confusion(matriks_r_persen, [f"{r} bintang" for r in label_rating], "Rating: persen per baris")

In [ ]:
selisih = (pd.Series(tebakan_rating, index=y_uji_r.index) - y_uji_r).abs()
salah = selisih[selisih > 0]
print(f"Tebakan yang salah: {len(salah):,} ({len(salah) / len(selisih):.0%})")
print(f"Dari yang salah, meleset hanya 1 bintang: {(salah == 1).mean():.0%}")

### Kesimpulan bagian B

Prediksi rating lima kelas jauh lebih sulit daripada sentimen dua kelas:

- Bintang 4 dan 5 hampir tidak bisa dibedakan dari teksnya. Pembeli yang memberi bintang 4 dan bintang 5 sering menulis kalimat yang sama, misalnya "barang bagus, pengiriman cepat". Perbedaan satu bintang itu lebih mencerminkan kebiasaan masing-masing pembeli daripada isi ulasannya.
- Bintang 2 dan 3 sangat sedikit dan isinya campuran, sehingga paling sering salah ditebak.
- Di antara bintang rendah, bintang 1 paling mudah dikenali (recall sekitar 62%), karena biasanya berisi keluhan yang jelas.
- Sebagian besar kesalahan hanya meleset satu bintang. Model memahami arah umum sentimennya, tetapi tidak bisa menebak angka pastinya.

Temuan ini mendukung keputusan di notebook 06 untuk menyederhanakan rating menjadi dua kelas (positif dan negatif). Dengan dua kelas, model menjawab pertanyaan yang memang bisa dijawab dari teks: apakah pembeli puas atau mengeluh.

## Ringkasan

| Tugas | Hasil | Catatan |
|---|---|---|
| Sentimen dua kelas (notebook 06) | F1 negatif sekitar 0,62 | Paling berguna dan dipakai di dashboard |
| Klasifikasi kategori | Akurasi sekitar 41% (patokan 39%), macro F1 sekitar 0,35 | Ulasan jarang menyebut produknya, dan model sebagian menghafal merek. Nama produk lebih informatif |
| Prediksi rating 1 sampai 5 | Macro F1 sekitar 0,38, tetapi akurasi dan MAE tidak lebih baik dari patokan | Bintang 4 dan 5 tertukar. Selisih antarbintang tidak tercermin di teks |

Pelajaran dari notebook ini: cara merumuskan pertanyaan sama pentingnya dengan pilihan model. Teks ulasan yang sama bisa menghasilkan model yang berguna (sentimen) atau kurang berguna (rating lima kelas), tergantung pertanyaan yang diajukan.